# Day 14 · Pandas 进阶

> 🎯 **今日学习目标**
> 1. 掌握布尔索引过滤与多条件筛选
> 2. 掌握 groupby 分组聚合
> 3. 掌握 merge/concat/pivot_table 数据合并

---

## 📖 引入：温故知新

Day 13 我们学了 `loc` 与 `iloc`，注意区别：
- `df.loc[1:3]` 按**标签**切片，**包含**末尾行
- `df.iloc[1:3]` 按**位置**切片，**不含**末尾行

今天进阶：**过滤、分组、合并** — 数据分析三板斧。

In [ ]:
# 导入今天的两个主力库
import pandas as pd   # 数据处理
import numpy as np    # 数值计算（制造缺失值会用到）

---

## 第一讲：过滤与排序

### 1.1 布尔索引

📖 **概念**：用布尔表达式生成 True/False mask，筛出满足条件的行。

In [ ]:
# 构造示例数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [25, 35, 28, 42],
    "城市": ["北京", "上海", "北京", "深圳"]
})

# 单条件：筛出年龄大于 25 的行
print(df[df["年龄"] > 25])

💡 **多条件：每个条件必须用括号！**

运算符 `&`（与）、`|`（或）优先级高于比较符，不加括号会报错。

In [ ]:
# 多条件：年龄 > 25 且 城市为北京
result = df[(df["年龄"] > 25) & (df["城市"] == "北京")]
print(result)

🔴 **常见错误**

```python
# 错误写法：缺少括号，优先级错乱
# df[df["年龄"] > 25 & df["城市"] == "北京"]  # 报错！
```

✅ 正确写法：每个条件都加括号。

### 1.2 query() 方法

📖 用**字符串表达式**写过滤条件，更简洁。

In [ ]:
# 等价于上面的多条件写法
result = df.query("年龄 > 25 and 城市 == '北京'")
print(result)

💡 用 `@` 引用外部变量：

In [ ]:
# 定义外部变量
min_age = 25
city = "北京"

# 在 query 中通过 @ 引用变量
print(df.query("年龄 > @min_age and 城市 == @city"))

### 1.3 排序

📖 `sort_values` 按值排序，支持多列、升降序。

In [ ]:
# 单列升序排序
print(df.sort_values("年龄"))

# 单列降序排序
print(df.sort_values("年龄", ascending=False))

In [ ]:
# 构造多列排序示例
df2 = pd.DataFrame({
    "班级": ["A", "A", "B", "B"],
    "成绩": [85, 92, 78, 92]
})

# 先按班级升序，再按成绩降序
print(df2.sort_values(
    by=["班级", "成绩"],
    ascending=[True, False]
))

---

### ✏️ 练习 1-1

对上面 `df` 数据，找出**年龄 ≥ 28 且不在北京**的同学。

In [ ]:
# TODO: 用布尔索引实现
pass

In [ ]:
# 参考答案
print(df[(df["年龄"] >= 28) & (df["城市"] != "北京")])

### ✏️ 练习 1-2

用 `query()` 找出**年龄 > 25 或城市为深圳**的行，并按年龄降序排列。

In [ ]:
# TODO: 用 query + sort_values 实现
pass

In [ ]:
# 参考答案
result = df.query("年龄 > 25 or 城市 == '深圳'")
result = result.sort_values("年龄", ascending=False)
print(result)

---

## 第二讲：分组聚合

### 2.1 groupby 基础

📖 **核心思想**：Split → Apply → Combine（拆分-应用-合并）

In [ ]:
# 构造学生成绩表
df = pd.DataFrame({
    "班级": ["A", "A", "B", "B", "A", "B"],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八"],
    "成绩": [85, 92, 78, 88, 90, 82]
})

In [ ]:
# 按班级分组，求每班平均成绩
result = df.groupby("班级")["成绩"].mean()
print(result)

In [ ]:
# 常用聚合函数：sum / max / min / count / std
print(df.groupby("班级")["成绩"].max())   # 最高分
print(df.groupby("班级")["成绩"].count()) # 人数

### 2.2 agg 多聚合

📖 `agg` 一次性计算**多种**聚合指标。

In [ ]:
# 同时求平均、最大、最小
result = df.groupby("班级")["成绩"].agg(["mean", "max", "min"])
print(result)

💡 **命名聚合语法**：为每列指定名字和聚合函数。

In [ ]:
# 命名聚合：输出列名更清晰
result = df.groupby("班级").agg(
    平均成绩=("成绩", "mean"),
    最高分=("成绩", "max"),
    人数=("成绩", "count")
)
print(result)

🔴 **常见错误**

`groupby` 之后要**指定列**再聚合，否则会对所有数值列都做：

In [ ]:
# 推荐：明确选中列
df.groupby("班级")["成绩"].mean()

# 不推荐：可能意外聚合所有数值列
# df.groupby("班级").mean(numeric_only=True)

---

### ✏️ 练习 2-3

根据上面 `df`，计算**每个班成绩的总和与标准差**。

In [ ]:
# TODO: 用 agg 实现
pass

In [ ]:
# 参考答案
result = df.groupby("班级")["成绩"].agg(["sum", "std"])
print(result)

### ✏️ 练习 2-4

用**命名聚合**语法，计算每班平均分、最低分、人数，
列名分别为：`平均分`、`最低分`、`人数`。

In [ ]:
# TODO: 用命名聚合实现
pass

In [ ]:
# 参考答案
result = df.groupby("班级").agg(
    平均分=("成绩", "mean"),
    最低分=("成绩", "min"),
    人数=("成绩", "count")
)
print(result)

---

## 第三讲：合并与透视

### 3.1 merge（SQL 风格 JOIN）

📖 按**键**合并两张表，类似 SQL 的 JOIN。

In [ ]:
# 构造两张表
df1 = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["张三", "李四", "王五"]
})

df2 = pd.DataFrame({
    "id": [1, 2, 4],
    "score": [85, 92, 78]
})

In [ ]:
# 内连接：只保留两边都有的 id（1 和 2）
inner = pd.merge(df1, df2, on="id")
print(inner)

In [ ]:
# 左连接：保留左表所有行，缺的填 NaN
left = pd.merge(df1, df2, on="id", how="left")
print(left)

In [ ]:
# 外连接：保留所有 id
outer = pd.merge(df1, df2, on="id", how="outer")
print(outer)

💡 **四种连接方式速记**

| how | 含义 |
|---|---|
| inner | 交集 |
| left | 左表全部 |
| right | 右表全部 |
| outer | 并集 |

### 3.2 concat（拼接）

📖 按轴方向**堆叠**数据，不依赖键。

In [ ]:
# 垂直拼接（纵向堆叠行）
df_a = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
df_b = pd.DataFrame({"x": [5, 6], "y": [7, 8]})

result = pd.concat([df_a, df_b], ignore_index=True)
print(result)

In [ ]:
# 横向拼接（按列合并）
df_c = pd.DataFrame({"z": [9, 10]})

result = pd.concat([df_a, df_c], axis=1)
print(result)

### 3.3 pivot_table（透视表）

📖 类似 Excel 透视表，按行列分组聚合。

In [ ]:
# 构造销售数据
sales = pd.DataFrame({
    "日期": ["周一", "周一", "周二", "周二"],
    "城市": ["北京", "上海", "北京", "上海"],
    "销量": [100, 150, 120, 130]
})

# 透视表：行=日期，列=城市，值=销量求和
result = sales.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum"
)
print(result)

🔴 **常见错误**

`pivot_table` 默认 aggfunc 是 `mean`，如果要做求和记得写 `aggfunc="sum"`。

---

### ✏️ 练习 3-5

有学生表 `students` 和班级表 `classes`，
用 **左连接** 合并，显示每个学生的班级名称。

In [ ]:
# 准备数据
students = pd.DataFrame({
    "sid": [1, 2, 3],
    "name": ["张三", "李四", "王五"],
    "class_id": [101, 102, 101]
})

classes = pd.DataFrame({
    "class_id": [101, 102],
    "class_name": ["火箭班", "实验班"]
})

# TODO: 用 merge 实现左连接
pass

In [ ]:
# 参考答案
result = pd.merge(
    students, classes,
    on="class_id",
    how="left"
)
print(result)

---

## 补充：缺失值处理

### 4.1 检测缺失值

📖 `isnull()` 返回布尔 DataFrame，True 表示 NaN。

In [ ]:
# 构造含缺失值的数据
df = pd.DataFrame({
    "A": [1, np.nan, 3],
    "B": [4, 5, np.nan]
})

# 检测缺失值
print(df.isnull())

In [ ]:
# 每列缺失值个数
print(df.isnull().sum())

### 4.2 删除缺失值

📖 `dropna()` 删除含 NaN 的行或列。

In [ ]:
# 删除含任意 NaN 的行
print(df.dropna())

# 只删除全为 NaN 的行
print(df.dropna(how="all"))

# 按列删除
# df.dropna(axis=1)

### 4.3 填充缺失值

📖 `fillna()` 用指定值或策略填充 NaN。

In [ ]:
# 用固定值 0 填充
print(df.fillna(0))

In [ ]:
# 用列均值填充（常见做法）
print(df.fillna(df.mean(numeric_only=True)))

In [ ]:
# 前向填充：用上一行的值
print(df.fillna(method="ffill"))
# 新版本也可写作：df.ffill()

💡 **三种填充策略对比**

| 策略 | 方法 | 适用场景 |
|---|---|---|
| 固定值 | `fillna(0)` | 明确默认值 |
| 均值 | `fillna(df.mean())` | 数值型，分布均匀 |
| 前向填充 | `ffill()` | 时间序列数据 |

---

## ⭐ 挑战题：综合实战

场景：三张表 — 订单、商品、用户。

**要求：**
1. 用 merge 合并订单 + 商品 + 用户
2. 对缺失金额用该商品均价填充
3. 按城市分组，统计总金额和订单数

In [ ]:
# 准备数据
orders = pd.DataFrame({
    "oid": [1, 2, 3, 4],
    "pid": [101, 102, 101, 103],
    "uid": [1, 2, 1, 3],
    "金额": [50, np.nan, 60, np.nan]
})

products = pd.DataFrame({
    "pid": [101, 102, 103],
    "商品": ["苹果", "香蕉", "橙子"]
})

users = pd.DataFrame({
    "uid": [1, 2, 3],
    "城市": ["北京", "上海", "北京"]
})

### ✏️ 挑战练习

In [ ]:
# TODO: 1. 合并三张表
# TODO: 2. 按商品均价填充缺失金额
# TODO: 3. 按城市分组统计总金额、订单数
pass

In [ ]:
# 参考答案

# 第 1 步：合并三张表
merged = orders.merge(products, on="pid").merge(users, on="uid")
print("合并后:")
print(merged)

# 第 2 步：按商品均价填充缺失金额
avg_by_product = merged.groupby("商品")["金额"].transform("mean")
merged["金额"] = merged["金额"].fillna(avg_by_product)
print("\n填充后:")
print(merged)

# 第 3 步：按城市分组统计
result = merged.groupby("城市").agg(
    总金额=("金额", "sum"),
    订单数=("金额", "count")
)
print("\n最终统计:")
print(result)

---

✅ **今日检查清单**

- [ ] 能用布尔索引做多条件过滤（记得加括号！）
- [ ] 能用 query() 写过滤条件
- [ ] 能用 groupby + agg 做多聚合
- [ ] 能用 merge 做内/左/外连接
- [ ] 能用 pivot_table 做透视表
- [ ] 能用 isnull / dropna / fillna 处理缺失值